# Change a model

Use `model` from {doc}`tutorials/notebooks/11-model-building` for these
examples. Make a copy when you want to compare model variants.

## Change a value

In [1]:
import jax
import jax.numpy as jnp
import numpy as np
import tensorflow_probability.substrates.jax.distributions as tfd

import liesel.model as lsl

In [2]:
# Python cell fragment: imports are visible in each guide.
x_data = jnp.linspace(-1.0, 1.0, 80)
y_data = 1.0 + 2.0 * x_data + 0.7 * jax.random.normal(jax.random.key(42), (80,))

prior_scale = lsl.Var.new_value(2.5, name="prior_scale")
beta = lsl.Var.new_param(
    jnp.zeros(2),
    dist=lsl.Dist(tfd.Normal, loc=0.0, scale=prior_scale),
    name="beta",
)
variance = lsl.Var.new_param(
    1.0,
    dist=lsl.Dist(tfd.LogNormal, loc=0.0, scale=0.5),
    name="variance",
)
x = lsl.Var.new_obs(x_data, name="x")

mu = lsl.Var.new_calc(
    lambda x, b: b[0] + b[1] * x,
    x,
    beta,
    name="mu",
)
sigma = lsl.Var.new_calc(jnp.sqrt, variance, name="sigma")
y = lsl.Var.new_obs(
    y_data,
    dist=lsl.Dist(tfd.Normal, loc=mu, scale=sigma),
    name="y",
)
model = lsl.Model(y)

# Match the tutorial after its reactive update.
beta.value = jnp.array([1.0, 2.0])


In [3]:
changed = model.copy()
changed.vars["beta"].value = jnp.array([0.5, 1.5])

In [4]:
np.asarray(changed.vars["mu"].value[:3]).round(3)

array([-1.   , -0.962, -0.924], dtype=float32)

In [5]:
round(float(changed.log_lik), 3)

-98.436

The mean and likelihood update automatically. The connections stay the same.
Retrieve variables from `changed` when editing the copy: a Python variable
that points to the original `beta` still belongs to the original model.

## Replace a prior

In [6]:
new_width = lsl.Var.new_value(1.0, name="new_prior_scale")
changed.vars["beta"].dist_node["scale"] = new_width

In [7]:
changed.vars["beta"].dist_node["scale"].name

'new_prior_scale'

The coefficient prior now depends on `new_prior_scale`. To replace the whole
prior, assign a new distribution node:

In [8]:
changed.vars["beta"].dist_node = lsl.Dist(tfd.Normal, loc=0.0, scale=1.5)
changed.rebuild_graph();

In [9]:
sorted(changed.vars)

['beta', 'mu', 'sigma', 'variance', 'x', 'y']

The old prior-scale variables disappear because they are no longer inputs of
the model's seed response `y`. `rebuild_graph()` rediscovers the graph from
its seeds; components added with `add_to_seeds=False` can also disappear.
{meth}`update() <liesel.model.Var.update>` instead recomputes outdated values without pruning the graph.

The same input indexing works for calculations. Here, `sigma` takes its
variance as its first positional input:

In [10]:
changed.vars["sigma"].value_node[0] = lsl.Var.new_value(0.5, name="fixed_variance")
changed.rebuild_graph();

In [11]:
round(float(changed.vars["sigma"].value), 3)

0.707

The response standard deviation is now about 0.707. This changes one connection.
It also removes the unused variance parameter and its prior from this model.

## Replace a variable

Start from another copy when comparing this change with the previous one:

In [12]:
fixed = model.copy()
fixed.replace("variance", lsl.Var.new_value(0.5, name="variance"));

In [13]:
sorted(fixed.parameters)

['beta']

Only `beta` remains a parameter. {meth}`~liesel.model.Model.replace` redirects
uses of the replaced variable throughout the model, whereas assigning one
{attr}`value_node <liesel.model.Var.value_node>` or {attr}`dist_node <liesel.model.Var.dist_node>` input changes just that connection. Both choices
change the statistical model here: variance is fixed and its prior is removed.